# 21 - vLLM cold prefill on the T4 (prefix caching OFF)
The first prefill run replayed the same 15 prompts with prefix caching on, so its TTFT was a cache hit. This notebook reruns it cold: `vllm serve --no-enable-prefix-caching`. Run on a **GPU T4 x2** session with Internet on and Persistence = Variables and Files, then press **Run All** (about 15-25 minutes, most of it installing vLLM). Do not use *Factory reset*.

In [ ]:
import os, sys, json, time, signal, shutil, subprocess, threading, glob

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working"
MODEL = f"{WORK}/models/Qwen2.5-1.5B-Instruct"
REPO = f"{WORK}/turingserve"
RESULTS = f"{WORK}/results"
PORT = 8001
ENGINE = "vllm"
LABEL = "t4-1.5b-prefill-cold"
os.makedirs(RESULTS, exist_ok=True)
os.environ["TS_KAGGLE"] = "1"
srv = None


def gpu_used_mib():
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                       capture_output=True, text=True)
    return int(r.stdout.strip().splitlines()[0]) if r.returncode == 0 and r.stdout.strip() else -1


def stop_server():
    global srv
    if srv is not None and srv.poll() is None:
        try:
            os.killpg(os.getpgid(srv.pid), signal.SIGTERM)
            srv.wait(timeout=30)
        except Exception:
            try:
                os.killpg(os.getpgid(srv.pid), signal.SIGKILL)
            except Exception:
                pass
    srv = None
    subprocess.run(["pkill", "-f", "vllm serve"], capture_output=True)
    time.sleep(3)
    for _ in range(30):
        if 0 <= gpu_used_mib() < 500:
            break
        time.sleep(2)
    print("server stopped, GPU 0 memory used:", gpu_used_mib(), "MiB")


def start_server(cmd, log_name):
    """Launch the server, wait for /health (up to 20 minutes), print the first completion."""
    global srv
    import requests
    stop_server()
    log_path = f"{WORK}/{log_name}"
    srv = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    for i in range(240):
        try:
            if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
                print(f"server up after {time.time() - t0:.0f}s")
                break
        except Exception:
            pass
        if srv.poll() is not None:
            print(open(log_path).read()[-3000:])
            raise RuntimeError("server exited during startup, log tail above")
        if i % 12 == 11:
            print(f"  still starting... {time.time() - t0:.0f}s")
        time.sleep(5)
    else:
        print(open(log_path).read()[-3000:])
        raise RuntimeError("server did not become healthy in 20 minutes")
    r = requests.post(f"http://localhost:{PORT}/v1/completions",
                      json={"model": "qwen", "prompt": "The capital of France is", "max_tokens": 16, "temperature": 0})
    print("test completion:", r.json()["choices"][0]["text"])


def run_bench(extra):
    """Run serve_bench, streaming its output so progress is visible."""
    cmd = [sys.executable, "-u", "-m", "benchmarks.serve_bench", "--base-url", f"http://localhost:{PORT}",
           "--engine", ENGINE, "--model", "qwen", "--tokenizer", MODEL, "--out", RESULTS, "--label", LABEL, *extra]
    p = subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, PYTHONUNBUFFERED="1"),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Token indices sequence length" in line or "Running this sequence" in line:
            continue
        print(line, end="", flush=True)
    p.wait()
    print("exit code", p.returncode)
    return p.returncode


print("setup ok; GPU 0 memory used:", gpu_used_mib(), "MiB")

## 1. Install vLLM and record versions

In [ ]:
def can_import(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not can_import("vllm"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError("pip install vllm failed")
# the pip warnings about numba, gradio, pillow and opentelemetry concern other preinstalled packages and are harmless
r = subprocess.run([sys.executable, "-c", "import vllm, torch; print(vllm.__version__, torch.__version__)"],
                   capture_output=True, text=True)
print("vllm / torch:", r.stdout.strip() or r.stderr[-1500:])
open(f"{WORK}/vllm_versions.txt", "w").write(r.stdout)

## 2. Model and benchmark client at tag `m3-rc`

In [ ]:
from huggingface_hub import snapshot_download

if not os.path.exists(f"{MODEL}/model.safetensors"):
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL)
print("model ok:", os.path.exists(f"{MODEL}/model.safetensors"))
if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "https://github.com/Muskanpandey0304/turingserve.git", REPO], check=True)
subprocess.run(["git", "fetch", "--tags", "--force"], cwd=REPO, check=True, capture_output=True)
subprocess.run(["git", "checkout", "-f", "m3-rc"], cwd=REPO, check=True, capture_output=True)
print("repo at:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "datasets", "transformers"], capture_output=True)

## 3. Start vLLM with prefix caching off

In [ ]:
start_server(["vllm", "serve", MODEL, "--served-model-name", "qwen", "--dtype", "half", "--max-model-len", "8192",
              "--gpu-memory-utilization", "0.85", "--port", str(PORT), "--no-enable-prefix-caching"], "vllm_prefill_cold.log")
print("--- prefix-caching setting reported by vLLM (must say False) ---")
import re
txt = open(f"{WORK}/vllm_prefill_cold.log").read()
print(set(re.findall(r"enable_prefix_caching=\w+", txt)))

## 4. Prefill benchmark (2048 / 4096 / 8000 tokens, 3 repetitions)

In [ ]:
run_bench(["--dataset", "prefill", "--num-prompts", "500", "--mode", "concurrency", "--concurrency", "1",
           "--repeat", "3", "--gpu-log"])

## 5. Per-length summary and download
Download **`/kaggle/working/vllm_prefill_cold_bundle.zip`**, then stop the session.

In [ ]:
stop_server()
files = sorted(glob.glob(f"{RESULTS}/*_{ENGINE}_prefill_{LABEL}.json"))
print("result files:", [os.path.basename(f) for f in files])
cold_ok = True
for f in files:
    d = json.load(open(f))
    r = d["runs"][0]
    print(f"\n{ENGINE}: {len(r['all'])} repetitions, {r['median']['num_requests']} requests each, errors {r['median']['errors']}")
    by = {}
    for q in r["requests"]:
        if q.get("first") and not q.get("error"):
            by.setdefault(q["prompt_tokens"], []).append((q["first"] - q["send"]) * 1000)
    print(f"{'prompt tokens':>14}{'n':>4}{'TTFT median ms':>16}{'min':>8}{'max':>8}{'prefill tok/s':>15}")
    for L in sorted(by):
        v = sorted(by[L])
        med = v[len(v) // 2]
        print(f"{L:>14}{len(v):>4}{med:>16.0f}{v[0]:>8.0f}{v[-1]:>8.0f}{L / (med / 1000):>15.0f}")
        if L >= 7000 and med < 300:
            cold_ok = False
if not cold_ok:
    print("\nWARNING: an 8k-token prompt came back in under 300 ms, which a T4 cannot do cold. Prefix caching is probably still on.")
else:
    print("\nsanity check passed: 8k-token TTFT is in the range of real cold prefill")

bundle = f"{WORK}/vllm_prefill_cold_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in files + glob.glob(f"{WORK}/vllm_prefill_cold.log") + glob.glob(f"{WORK}/vllm_versions.txt"):
    shutil.copy(f, bundle)
shutil.make_archive(bundle, "zip", bundle)
print("\nDownload this one file:", bundle + ".zip", f"({os.path.getsize(bundle + '.zip') / 1e6:.1f} MB)")